# EffoVPR_G - local place label prediction

Use the local model artifact in `Models/EffoVPR_G/` to classify every image in `Images/`. This notebook does not download a gallery or contact Hugging Face. The output is the predicted attraction label for each image; the cosine score is a ranking score, not a calibrated probability.


## Dependencies (run once if needed)

The artifact was exported with Transformers 5.18.0. Install compatible packages in the active Jupyter environment if they are missing.


In [9]:
# Uncomment and run if dependencies are missing:
# %pip install "transformers==5.18.0" "safetensors>=0.4.3" "pandas>=2.0" "Pillow>=10.0"


## 1. Configure local paths

In [10]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src/effovpr").is_dir() and (candidate / "Models").is_dir():
        ROOT = candidate
        break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / "Models" / "EffoVPR_G"
IMAGE_DIR = ROOT / "Images"
OUTPUT_CSV = ROOT / "artifacts" / "results" / "local_place_predictions.csv"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

for name in ("config.json", "model.safetensors"):
    if not (MODEL_DIR / name).is_file():
        raise FileNotFoundError(f"Missing model file: {MODEL_DIR / name}")
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}")
print("Model:", MODEL_DIR)
print("Images:", IMAGE_DIR)

Model: D:\Github\EffoVPR\Models\EffoVPR_G
Images: D:\Github\EffoVPR\Images


## 2. Load the model

In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from src.effovpr.utils.model_artifact import load_model_artifact

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, artifact = load_model_artifact(MODEL_DIR, DEVICE)
resolution = int(artifact["inference"]["resolution"])
class_mapping = artifact.get("class_mapping") or {}
id_to_label = {int(class_id): label for label, class_id in class_mapping.items()}
if model.cosface is None or not id_to_label:
    raise ValueError("This artifact has no CosFace classifier or class mapping.")
print("Device:", DEVICE, "| input resolution:", resolution, "| classes:", len(id_to_label))

Device: cpu | input resolution: 504 | classes: 49


## 3. Predict one place label for every image in `Images/`

In [13]:
@torch.inference_mode()
def predict_place(image):
    image = image.convert("RGB").resize((resolution, resolution), Image.Resampling.BICUBIC)
    pixel_values = model.backbone.image_processor(
        images=[image], do_resize=False, do_center_crop=False, return_tensors="pt"
    )["pixel_values"].to(DEVICE)
    embedding = model.extract_global(pixel_values).float()
    # CosFace inference uses cosine similarity to normalized class weights.
    class_weights = F.normalize(model.cosface.weight.float(), dim=1)
    class_scores = F.normalize(embedding, dim=1) @ class_weights.T
    class_id = int(class_scores.argmax(dim=1).item())
    return id_to_label[class_id], float(class_scores[0, class_id].item())

image_paths = sorted(p for p in IMAGE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
if not image_paths:
    raise ValueError(f"No supported images found in {IMAGE_DIR}")

rows = []
for path in image_paths:
    try:
        with Image.open(path) as image:
            label, score = predict_place(image)
        rows.append({"image": str(path.relative_to(ROOT)), "predicted_label": label, "cosine_score": score})
    except Exception as exc:
        print("Skipping unreadable image:", path, exc)

if not rows:
    raise ValueError("No images could be classified.")
predictions = pd.DataFrame(rows)
display(predictions)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_CSV)

,image,predicted_label,cosine_score
0,Images\Hang Son Doong.jpg,Hang Sơn Đoòng,0.790497
1,Images\img_013.jpg,Chợ Bến Thành,0.841543
2,Images\img_022.jpg,Ngũ Hành Sơn,0.714569
3,Images\NguHanhSon1.jpg,Ngũ Hành Sơn,0.694872


Saved: D:\Github\EffoVPR\artifacts\results\local_place_predictions.csv
